# u4lib — Python tutorial

Bindings for the header-only C++ **u4lib** — Wigner-Racah algebra for U(4)/SU(4),
the Wigner **supermultiplet** symmetry (spin-isospin SU(4) ⊃ SU(2)_S × SU(2)_T).
Names mirror the C++ API 1:1. Build first with `make python`; the bindings are
in [pyu4lib.cpp](pyu4lib.cpp).

| Python entry point | Purpose |
|---|---|
| `su4.irrep(P1,P2,P3)`, `irrep(P1,P2,P3)`, `irrep(n1,n2,n3,n4)` | SU(4)/U(4) irreps |
| `gelfand(...)` | Gelfand–Tsetlin state (10 labels) + generator MEs |
| `physicalvector(irrep, SS, TT)` | physical (S,T) state |
| `Eta(a, b, c)` | Coupling multiplicity of c in a×b |
| `Zeta(irrep, SS, TT)` | (S,T) branching multiplicity (SS=2S, TT=2T) |
| `physicalBasis`, `gelfandBasis` | basis state enumerators |
| `wigU4U3db().get(...)` | U(4) ⊃ U(3) coefficients (2-D numpy) |
| `wigU4SU2SU2db().get_U4SU2SU2(...)` | U(4) ⊃ SU(2)×SU(2), structured |
| `u4u6db()/u4z6db()/u4u9db().get_U4*(...)` | 6-/9-irrep recoupling |

> **Asserts are compiled in** (no `-DNDEBUG`): indexing an empty/forbidden
> coupling `abort()`s the kernel. Guard with `.empty()` or check `Eta`/`Zeta`.

## 1. Import + initialize
`initialize()` cascades u4 → u3 → qcx_utils (Wigner backend + all DBs).

In [1]:
from pathlib import Path
import sys
import numpy as np
from IPython.display import Markdown, display

candidates = []
for base in (Path.cwd(), *Path.cwd().parents):
    candidates.extend((base, base / "python", base / "u4lib" / "python"))
python_dir = next((p.resolve() for p in candidates
                   if (p / "pyu4lib.cpp").is_file()), None)
if python_dir is None:
    raise FileNotFoundError("Launch Jupyter from QCX, u4lib, or u4lib/python.")
sys.path.insert(0, str(python_dir))
import u4lib as u4
np.set_printoptions(precision=10, suppress=True)
print("Extension:", u4.__file__)

def table(headers, rows):
    lines = ["| " + " | ".join(map(str, headers)) + " |",
             "| " + " | ".join(["---"] * len(headers)) + " |"]
    lines += ["| " + " | ".join(map(str, r)) + " |" for r in rows]
    display(Markdown("\n".join(lines)))

u4.initialize()
print("u4lib session initialized.")

Extension: /Users/phongdang/Desktop/QCX/u4lib/python/u4lib.so
u4lib session initialized.


## 2. Irreps

- `su4.irrep(P1, P2, P3)` — an SU(4) irrep; `str()` → `(P1,P2,P3)`.
- `irrep(P1, P2, P3)` — a U(4) irrep from SU(4) labels (n44 = 0); `str()` → `[n14,n24,n34,n44]`.
- `irrep(n1, n2, n3, n4)` — a U(4) irrep from the four row labels.

Dimensions (spot-checks): fundamental `(1,0,0)` = **4**, `(2,0,0)` = **10**,
`(0,1,0)` = **6**, adjoint `(1,0,1)` = **15** (self-conjugate), `(0,0,0)` = **1**.

In [2]:
labels = [(0,0,0), (1,0,0), (2,0,0), (0,1,0), (1,0,1)]
rows = []
for P in labels:
    ir = u4.irrep(*P)
    rows.append((f"({P[0]},{P[1]},{P[2]})", str(ir),
                 (ir.n14(), ir.n24(), ir.n34(), ir.n44()), ir.dim(), str(ir.conjugate()),
                 round(ir.C1(),6), round(ir.C2(),6), round(ir.C3(),6), round(ir.C4(),6)))
table(["SU(4) (P1,P2,P3)", "U(4) str", "(n14..n44)", "dim", "conjugate", "C1", "C2", "C3", "C4"], rows)

# Two ways to build the same U(4) irrep:
print("irrep(2,0,0)   ->", str(u4.irrep(2,0,0)))            # [2,0,0,0]
print("irrep(4,2,2,0) ->", str(u4.irrep(4,2,2,0)),
      " (P1,P2,P3) =", u4.irrep(4,2,2,0).P1(), u4.irrep(4,2,2,0).P2(), u4.irrep(4,2,2,0).P3())

adj = u4.su4.irrep(1,0,1)
print("adjoint (1,0,1): dim", adj.dim(), "| self-conjugate?", str(adj.conjugate()) == str(adj),
      "| C2", adj.C2(), "| C3", adj.C3(), "| C4", adj.C4())

| SU(4) (P1,P2,P3) | U(4) str | (n14..n44) | dim | conjugate | C1 | C2 | C3 | C4 |
| --- | --- | --- | --- | --- | --- | --- | --- | --- |
| (0,0,0) | [0,0,0,0] | (0, 0, 0, 0) | 1 | [0,0,0,0] | 0 | 0.0 | 0.0 | 0.0 |
| (1,0,0) | [1,0,0,0] | (1, 0, 0, 0) | 4 | [1,1,1,0] | 1 | 7.5 | 22.5 | 26.953125 |
| (2,0,0) | [2,0,0,0] | (2, 0, 0, 0) | 10 | [2,2,2,0] | 2 | 18.0 | 72.0 | 92.25 |
| (0,1,0) | [1,1,0,0] | (1, 1, 0, 0) | 6 | [1,1,0,0] | 2 | 10.0 | 0.0 | 26.25 |
| (1,0,1) | [2,1,1,0] | (2, 1, 1, 0) | 15 | [2,1,1,0] | 4 | 16.0 | 0.0 | 72.0 |

irrep(2,0,0)   -> [2,0,0,0]
irrep(4,2,2,0) -> [4,2,2,0]  (P1,P2,P3) = 2 0 2
adjoint (1,0,1): dim 15 | self-conjugate? True | C2 16.0 | C3 0.0 | C4 72.0


## 3. Gelfand–Tsetlin states

A U(4) GT state with 10 labels `n14,n24,n34,n44; n13,n23,n33; n12,n22; n11`.
Exposes the betweenness `check()`, SU(2) labels (2Λ, 2M_Λ), and the U(4)
generator matrix elements (`e11`…`e44` and the multi-step raising/lowering ones).

In [3]:
g = u4.gelfand(1,0,0,0, 1,0,0, 1,0, 1)   # a state of the fundamental (1,0,0)
print("state     :", str(g), "| repr", repr(g))
print("valid?    :", g.check())
print("P1,P2,P3  :", g.P1(), g.P2(), g.P3())
print("2*Lambda, 2*M_Lambda:", g.Lambdax2(), g.MLambdax2())
print("e11,e22,e33,e44:", g.e11(), g.e22(), g.e33(), g.e44())
print("conjugate :", str(g.conjugate()), "| phase", g.conjugate_phase())

state     : 1,0,0,0;1,0,0;1,0;1 | repr gelfand(1,0,0,0;1,0,0;1,0;1)
valid?    : True
P1,P2,P3  : 1 0 0
2*Lambda, 2*M_Lambda: 1 1
e11,e22,e33,e44: 1.0 0.0 0.0 0.0
conjugate : 1,1,1,0;1,1,0;1,0;0 | phase 1.0


## 4. Coupling multiplicity `Eta`

`Eta(a, b, c)` = multiplicity of `c` in `a × b` (u4's analog of u3's `Rho`).
Two overloads: SU(4) labels, or irrep objects.

Spot-check: `4 ⊗ 4 = 10 ⊕ 6` — i.e. `(1,0,0)×(1,0,0)` gives `(2,0,0)` and
`(0,1,0)`, each once, and the dimensions sum to 16.

In [4]:
fund = u4.irrep(1,0,0)
outs = [(2,0,0), (0,1,0), (0,0,0)]
rows = []
for P in outs:
    c = u4.irrep(*P)
    e = u4.Eta(fund, fund, c)
    rows.append((f"({P[0]},{P[1]},{P[2]})", c.dim(), e, c.dim()*e))
table(["output", "dim", "Eta", "dim x Eta"], rows)
tot = sum(r[-1] for r in rows if r[0] != "(0,0,0)")
print("4 x 4: (2,0,0) + (0,1,0) dims =", tot, "= 16 = 4*4:", tot == 16)
print("Eta by labels == by objects:",
      u4.Eta(1,0,0, 1,0,0, 2,0,0), u4.Eta(fund, fund, u4.irrep(2,0,0)))

| output | dim | Eta | dim x Eta |
| --- | --- | --- | --- |
| (2,0,0) | 10 | 1 | 10 |
| (0,1,0) | 6 | 1 | 6 |
| (0,0,0) | 1 | 0 | 0 |

4 x 4: (2,0,0) + (0,1,0) dims = 16 = 16 = 4*4: True
Eta by labels == by objects: 1 1


## 5. Spin-isospin content: `Zeta` and the physical basis

`Zeta(irrep, SS, TT)` = multiplicity of the spin-isospin channel `(S,T)` in the
U(4) irrep, with **SS = 2S** and **TT = 2T** (doubled).

Spot-check: the fundamental `(1,0,0)` is the **nucleon** supermultiplet
`(S,T) = (½,½)` → `SS=TT=1`, `Zeta = 1`; the symmetric `(2,0,0)` ("10") contains
`(S,T) = (1,1)` and `(0,0)` → `(SS,TT) = (2,2)` and `(0,0)` (that is 9 + 1 = 10).

In [5]:
print("Zeta((1,0,0), SS=1, TT=1) [nucleon (1/2,1/2)]:", u4.Zeta(1,0,0, 1,1))
print("physicalBasis((1,0,0)) (SS,TT,zeta):",
      [(v.SS(), v.TT(), v.zeta()) for v in u4.physicalBasis(u4.irrep(1,0,0))])

# physicalBasis as a table -- the (SS,TT,zeta) content of an irrep:
pb = u4.physicalBasis(u4.irrep(2,0,0))
table(["state", "SS (=2S)", "TT (=2T)", "zeta"],
      [(repr(v), v.SS(), v.TT(), v.zeta()) for v in pb])

# gelfandBasis size == dim of the irrep
gb = u4.gelfandBasis(u4.irrep(0,1,0))
print("gelfandBasis((3,1,0)):", len(gb), "states (= dim", u4.irrep(0,1,0).dim(), ")")
table(["state"], [(repr(g),) for g in gb])

Zeta((1,0,0), SS=1, TT=1) [nucleon (1/2,1/2)]: 1
physicalBasis((1,0,0)) (SS,TT,zeta): [(1, 1, 1)]


| state | SS (=2S) | TT (=2T) | zeta |
| --- | --- | --- | --- |
| physicalvector((2,0,0), zeta=1, SS=2, TT=2) | 2 | 2 | 1 |
| physicalvector((2,0,0), zeta=1, SS=0, TT=0) | 0 | 0 | 1 |

gelfandBasis((3,1,0)): 6 states (= dim 6 )


| state |
| --- |
| gelfand(1,1,0,0;1,1,0;1,1;1) |
| gelfand(1,1,0,0;1,1,0;1,0;1) |
| gelfand(1,1,0,0;1,1,0;1,0;0) |
| gelfand(1,1,0,0;1,0,0;1,0;1) |
| gelfand(1,1,0,0;1,0,0;1,0;0) |
| gelfand(1,1,0,0;1,0,0;0,0;0) |

## 6. U(4) ⊃ U(3) coefficients — `wigU4U3db`

`get(...)` takes, per irrep a/b/c, the seven U(4)⊃U(3) labels
`(n14, n24, n34, n44, n13, n23, n33)`, and returns a **2-D** numpy array indexed
`[rho, eta]` (rho = SU(3) coupling multiplicity, eta = U(4) coupling multiplicity).

In [9]:
m = np.asarray(u4.wigU4U3db().get_U4U3(2,1,1,0,2,1,0 , 3,2,1,0,3,2,0 , 3,2,1,0,3,2,0))
print(m)

rho	eta
1	1	0.331286
1	2	0.753849
1	3	-0.0670931
2	1	0.104328
2	2	0.0239579
2	3	0.866283



## 7. U(4) ⊃ SU(2)×SU(2) coefficients — `wigU4SU2SU2db`

Per irrep: `(P1, P2, P3, SS, TT)`. `get(...)` returns a flat numpy array;
`get_U4SU2SU2(...)` returns a structured **U4SU2SU2** object indexed
`(zetaA, zetaB, zetaC, eta)`.

In [11]:
db = u4.wigU4SU2SU2db()
# scalar check:
sc = db.get_U4SU2SU2(0,0,0,0,0, 0,0,0,0,0, 0,0,0,0,0)
print("scalar (0,0,0): size", sc.size(), "data", sc.data())

obj = db.get_U4SU2SU2(1,1,1,2,2,  1,1,1,4,2,  1,2,1,4,2)
print("empty?", obj.empty(), "| shape (zA,zB,zC,eta) =",
      obj.zetaAmax(), obj.zetaBmax(), obj.zetaCmax(), obj.etamax())
print(".data() =", obj.data())
print(obj)

scalar (0,0,0): size 1 data [1.]
empty? False | shape (zA,zB,zC,eta) = 2 1 2 4
.data() = [-0.3354101966  0.125        -0.2201607247 -0.1353740386  0.0968245837
 -0.1804219591  0.2612813658  0.1898128628  0.2515576475 -0.09375
 -0.0795024839  0.0773565935  0.2097865979 -0.2706329387  0.0717935284
  0.253083817 ]
zetaA	zetaB	zetaC	eta
1	1	1	1	 -> -0.33541
1	1	1	2	 -> 0.125
1	1	1	3	 -> -0.220161
1	1	1	4	 -> -0.135374
1	1	2	1	 -> 0.0968246
1	1	2	2	 -> -0.180422
1	1	2	3	 -> 0.261281
1	1	2	4	 -> 0.189813
2	1	1	1	 -> 0.251558
2	1	1	2	 -> -0.09375
2	1	1	3	 -> -0.0795025
2	1	1	4	 -> 0.0773566
2	1	2	1	 -> 0.209787
2	1	2	2	 -> -0.270633
2	1	2	3	 -> 0.0717935
2	1	2	4	 -> 0.253084



## 8. Recoupling — `u4u6db` / `u4z6db` / `u4u9db`

The 6-irrep U- and Z-coefficients and the 9-irrep coefficients. Each `get_U4*`
takes the relevant coupling **multiplicities** (from `Eta`) and returns a
structured object. A scalar leg keeps every multiplicity 1.

In [15]:
A=u4.irrep(2,1,1,0); B=u4.irrep(2,1,1,0); C=u4.irrep(2,1,1,0)
AB=u4.irrep(3,1,0,0); BC=u4.irrep(2,1,1,0); AC=u4.irrep(2,2,0,0)
ABC=u4.irrep(3,1,0,0)

eAB,eAB_C = u4.Eta(A,B,AB), u4.Eta(AB,C,ABC)
eBC,eA_BC = u4.Eta(B,C,BC), u4.Eta(A,BC,ABC)
eAC,eAC_B = u4.Eta(A,C,AC), u4.Eta(AC,B,ABC)

u6 = u4.u4u6db().get_U4U6(A,B,ABC,C, AB,eAB,eAB_C, BC,eBC,eA_BC)
print("U4U6 size", u6.size())
print(u6)

z6 = u4.u4z6db().get_U4Z6(B,A,ABC,C, AB,eAB,eAB_C, AC,eAC,eAC_B)
print("U4Z6 size", z6.size())
print(z6)

U4U6 size 4
etaAB	etaAB_C	etaBC	etaA_BC
1	1	1	1	 -> 0.625
1	1	2	1	 -> -0.0721688
1	2	1	1	 -> 0.330719
1	2	2	1	 -> 0.572822

U4Z6 size 2
etaAB	etaAB_C	etaAC	etaAC_B
1	1	1	1	 -> -0.53033
1	2	1	1	 -> 0.467707



In [17]:
A=u4.irrep(2,1,1,0); B=u4.irrep(2,1,1,0); C=u4.irrep(2,1,1,0); D=u4.irrep(2,1,1,0)
AB=u4.irrep(2,1,1,0); CD=u4.irrep(2,1,1,0); ACi=u4.irrep(2,1,1,0); BD=u4.irrep(2,1,1,0)
ABCD=u4.irrep(0,0,0,0)
eAB,eCD = u4.Eta(A,B,AB), u4.Eta(C,D,CD)
eAC,eBD = u4.Eta(A,C,ACi), u4.Eta(B,D,BD)
eABCD,eACBD = u4.Eta(AB,CD,ABCD), u4.Eta(ACi,BD,ABCD)
u9 = u4.u4u9db().get_U4U9(A,B,C,D, AB,CD, ACi,BD, ABCD, eAC,eBD,eABCD, eAB,eCD,eACBD)
print("U4U9 size", u9.size())
print(u9)

U4U9 size 16
etaAC	etaBD	etaABCD	etaAB	etaCD	etaACBD
1	1	1	1	1	1	 -> -0.0625
1	1	1	1	2	1	 -> -0.108253
1	1	1	2	1	1	 -> -0.108253
1	1	1	2	2	1	 -> -0.1875
1	2	1	1	1	1	 -> -0.108253
1	2	1	1	2	1	 -> -0.1875
1	2	1	2	1	1	 -> 0.8125
1	2	1	2	2	1	 -> 0.252591
2	1	1	1	1	1	 -> -0.108253
2	1	1	1	2	1	 -> 0.8125
2	1	1	2	1	1	 -> -0.1875
2	1	1	2	2	1	 -> 0.252591
2	2	1	1	1	1	 -> -0.1875
2	2	1	1	2	1	 -> 0.252591
2	2	1	2	1	1	 -> 0.252591
2	2	1	2	2	1	 -> 0.104167



## 9. Finish

In [18]:
u4.finalize()
print("Session finalized. All tutorial checks passed.")

Session finalized. All tutorial checks passed.
